# Notebook 03 : FINAL, phase 6 (les deux ensemble)
**Auteurs : ........ et ........**

Rejeu de l'attaque contre le modèle durci, autre attaque, stabilité, résultats finaux sur le test.

**Avant de commencer** :
1. A a exécuté `01_defenseur.ipynb` en entier (donne `artifacts/detectors.joblib` et `artifacts/hardened.joblib`).
2. B a exécuté `02_attaquant.ipynb` jusqu'à la section 4.2 (donne `results/attaque_gloutonne_avant.csv`).
3. Une seule personne lance ce notebook pour les résultats finaux, de haut en bas.

## Chargement

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import matplotlib.pyplot as plt
from src.common import *
from src.attack import *
import src.attack as atk

S = load_state()
DET, BEST = load_detectors()
hardened = load_hardened()
atk.set_bounds(S.hi)
target = DET[TARGET_NAME]
curve_orig = pd.read_csv("results/attaque_gloutonne_avant.csv")      # produit par le notebook 02
ATT_VAL = sample_phish(S.urls_val, S.y_val, N_ATTACK, SEED)          # même échantillon que le notebook 02
print("détecteur :", target.name, "| durci :", hardened.name, "| échantillon d'attaque :", len(ATT_VAL))

## 6.1 Rejeu de l'attaque gloutonne contre le modèle durci (mêmes échantillons, mêmes budgets)

In [ ]:
curve_hard, store_hard = run_attack(hardened, ATT_VAL, BUDGETS, "greedy")
curve_hard.to_csv("results/attaque_gloutonne_apres.csv", index=False)
both = curve_orig[["budget", "rappel_sous_attaque", "taux_evasion"]].merge(
    curve_hard[["budget", "rappel_sous_attaque", "taux_evasion"]], on="budget", suffixes=("_avant", "_apres"))
both

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot(curve_orig.budget, curve_orig.rappel_sous_attaque, "o-", label="avant durcissement")
ax.plot(curve_hard.budget, curve_hard.rappel_sous_attaque, "s-", label="après durcissement")
ax.set_xlabel("Amplitude autorisée (coût total)"); ax.set_ylabel("Rappel sous attaque"); ax.set_ylim(0, 1.02)
ax.set_title("Rappel vs amplitude, avant et après durcissement (validation)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/05_rappel_vs_amplitude_avant_apres.png", dpi=130); plt.show()

## 6.2 Le durcissement résiste-t-il à une attaque **différente** ?
Attaque aléatoire (jamais vue par le modèle durci) et budgets plus grands que celui du durcissement.

In [ ]:
BIG = [B_REF, 12, 20]
cA, _ = run_attack(target,   ATT_VAL, BIG, "random")
cB, _ = run_attack(hardened, ATT_VAL, BIG, "random")
cC, _ = run_attack(target,   ATT_VAL, BIG, "greedy")
cD, _ = run_attack(hardened, ATT_VAL, BIG, "greedy")
other = pd.DataFrame({"budget": BIG,
                      "évasion aléatoire avant": cA.taux_evasion, "évasion aléatoire après": cB.taux_evasion,
                      "évasion gloutonne avant": cC.taux_evasion, "évasion gloutonne après": cD.taux_evasion})
other.to_csv("results/autres_attaques.csv", index=False); other

## 6.3 Stabilité : 3 tirages d'échantillons d'attaque (moyenne ± écart-type)

In [ ]:
rows = []
for sd in (0, 1, 2):
    S_ = sample_phish(S.urls_val, S.y_val, N_ATTACK, SEED + sd)
    for d in (target, hardened):
        c, _ = run_attack(d, S_, [B_REF], "greedy", seed=sd)
        rows.append(dict(modèle=d.name, graine=sd, taux_evasion=c.taux_evasion.iloc[0], rappel_sous_attaque=c.rappel_sous_attaque.iloc[0]))
seeds_df = pd.DataFrame(rows)
stab = seeds_df.groupby("modèle")[["taux_evasion", "rappel_sous_attaque"]].agg(["mean", "std"]).round(4)
stab.to_csv("results/stabilite_3_graines.csv"); stab

## 6.5 Résultats finaux sur le **test** (une seule exécution, aucune décision n'est modifiée après)

In [ ]:
ATT_TEST = sample_phish(S.urls_test, S.y_test, N_ATTACK, SEED)
final_normal = pd.DataFrame([summary_row(d.name, S.y_test, d.score_F(S.X_test), d.thr) for d in (target, hardened)]).set_index("modèle")
final_normal.to_csv("results/test_performance_normale.csv")
display(final_normal)

ct_o, _ = run_attack(target,   ATT_TEST, BUDGETS, "greedy")
ct_h, _ = run_attack(hardened, ATT_TEST, BUDGETS, "greedy")
ct_o.to_csv("results/test_attaque_avant.csv", index=False); ct_h.to_csv("results/test_attaque_apres.csv", index=False)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot(ct_o.budget, ct_o.rappel_sous_attaque, "o-", label="avant durcissement")
ax.plot(ct_h.budget, ct_h.rappel_sous_attaque, "s-", label="après durcissement")
ax.set_xlabel("Amplitude autorisée (coût total)"); ax.set_ylabel("Rappel sous attaque"); ax.set_ylim(0, 1.02)
ax.set_title("Résultat final sur le jeu de test"); ax.legend()
plt.tight_layout(); plt.savefig("figures/06_test_final_rappel_vs_amplitude.png", dpi=130); plt.show()
pd.concat({"avant": ct_o.set_index("budget")[["rappel_sous_attaque", "taux_evasion", "requetes_mediane"]],
           "après": ct_h.set_index("budget")[["rappel_sous_attaque", "taux_evasion", "requetes_mediane"]]}, axis=1)

> **À écrire dans le rapport (phase 6 et conclusion)** :
> 1. La figure finale avant/après (validation et test) et les chiffres au budget `B_REF`.
> 2. Le durcissement a-t-il aidé contre l'attaque connue ? contre l'attaque aléatoire et les budgets plus grands (6.2) ? de combien, avec quelle stabilité (6.3) ?
> 3. Le prix en performance normale (notebook 01 section 5.2, et test ci-dessus).
> 4. Le transfert (notebook 02, 6.4) : si l'évasion transférée reste proche de l'évasion directe, cacher le modèle protège peu.
> 5. **Limites honnêtes** : variables approchées (domaine = deux derniers labels), attaque uniquement dans l'espace des variables d'URL, adaptation de l'attaquant limitée à 2 tours de durcissement,
>    un seul jeu de données avec ses biais de collecte, seuil fixé sur une validation, pas de test sur de vrais utilisateurs.
> 6. Réalisme du modèle de menace : un vrai attaquant connaît rarement le score exact et peut aussi changer de domaine ou de contenu.